# Prepare Raw Data for SHAP Analysis

This notebook preprocesses raw data exactly as done in trabalho3.ipynb and exports it for SHAP analysis.
Output: `data/prepared_data_for_shap.csv` (21 engineered features)

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import importlib

# Force reload
import data_preparation_final
importlib.reload(data_preparation_final)

from data_preparation_final import load_and_clean_data

## 2. Load and Clean Raw Data

In [ ]:
def prepare_data_for_lgbm(df):
    """
    Cleaning and filtering for LightGBM model.
    (Same as in trabalho3.ipynb)
    """
    print("\n--- Data Preparation for LightGBM ---")
    inicial = len(df)
    
    # 1. Critical cleaning: dropna on fundamental fields
    cols_limpeza = ['dt_despacho_pedido', 'dt_entrega_pedido', 'dt_pagamento_pedido', 'qtd_dias_tat']
    df = df.dropna(subset=cols_limpeza).copy()
    
    posterior = len(df)
    print(f"Records before: {inicial}")
    print(f"Records after cleaning (dropna): {posterior}")
    print(f"Data loss: {1 - (posterior/inicial):.2%}")

    # 2. Outlier treatment (99th percentile on qtd_dias_tat)
    limite_99 = df['qtd_dias_tat'].quantile(0.99)
    df = df[df['qtd_dias_tat'] <= limite_99].copy()
    print(f"Outliers removed (TAT > {limite_99:.1f} days): {posterior - len(df)}")

    return df

# Load and clean
input_file = "pedidos_logistica.parquet"
print(f"Loading {input_file}...")
df = load_and_clean_data(input_file, drop_ids=False)

if df is not None:
    df = prepare_data_for_lgbm(df)
    print(f"\nData shape after preprocessing: {df.shape}")
else:
    print("ERROR: Failed to load data")
    df = None

In [ ]:
# Drop unnecessary columns
df = df.drop(
    columns=[
        'dt_entrega_pedido',
        'flg_existem_ocorrencias'
    ],
    errors='ignore'
)

print(f"Columns after dropping: {list(df.columns)}")

## 3. Feature Engineering

In [ ]:
# --- 1. Temporal features from dispatch date ---
df['dia_semana_despacho'] = df['dt_despacho_pedido'].dt.dayofweek
df['mes_despacho']        = df['dt_despacho_pedido'].dt.month
df['hora_despacho'] = df['dt_despacho_pedido'].dt.hour
df['semana_ano']          = df['dt_despacho_pedido'].dt.isocalendar().week.astype('float32')

# Weekend dispatch flag (Sat=5, Sun=6)
df['is_fds_despacho']   = (df['dia_semana_despacho'] >= 5).astype('int8')

# High-demand season: November (Black Friday) and December (Christmas)
df['is_alta_temporada'] = df['mes_despacho'].isin([11, 12]).astype('int8')

# Dispatch shift
if 'hora_despacho' in df.columns:
    df['turno_despacho'] = pd.cut(
        df['hora_despacho'],
        bins=[-1, 5, 11, 17, 23],
        labels=['Madrugada', 'Manha', 'Tarde', 'Noite']
    ).astype('category')

# --- 2. Derived date-ratio features ---
df['dias_criacao_pagamento'] = (df['dt_pagamento_pedido'] - df['dt_criacao']).dt.days

# Days the carrier has from dispatch to expected delivery
df['prazo_apos_despacho'] = (
    df['dt_previsao_entrega_cliente'] - df['dt_despacho_pedido']
).dt.days

# Fraction of total deadline already consumed inside the CD (>1 = deadline already blown)
df['ratio_cd_prazo'] = (
    df['dias_gastos_cd'] / df['dias_restantes_prazo'].replace(0, np.nan)
).clip(0, 2)

# Net delivery margin: negative = deadline blown before dispatch
df['margem_entrega'] = df['prazo_apos_despacho'] - df['dias_gastos_cd']

print("\nFeature Engineering Complete!")
print(f"Total features: {len(df.columns)}")
print(f"Data shape: {df.shape}")

## 4. Export Preprocessed Data for SHAP

In [ ]:
# Export preprocessed data for SHAP
output_path = "data/prepared_data_for_shap.csv"

print(f"\nExporting preprocessed data to: {output_path}")
df.to_csv(output_path, index=False)

print(f"✓ Success!")
print(f"  File: {output_path}")
print(f"  Rows: {len(df)}")
print(f"  Columns ({len(df.columns)}): {', '.join(df.columns)}")
print(f"\nNow run SHAP with:")
print(f"  python shap_updated.py --input {output_path}")